# Lab 5.2 &mdash; Decomposition, Distribution and Handoffs

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 2 &middot; Module 5 &mdash; Multi-Agent Collaboration &amp; Orchestration**

### What you'll do
- Turn a dependency graph into execution waves &mdash; what may run at once, and what may not
- Thread state through a run and watch the critical path refuse to get shorter
- Build the handoff payload, and decide what crosses it
- Reproduce the bug where both agents are right and the answer is wrong

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **Builds on Lab 5.1's supervisor.** Routing picked <em>who</em>. This lab is about
> <em>in what order</em>, and what each one is told when its turn comes.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-5-02")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 5 labs -- the same payment exceptions, now worked
# by several agents at once, and finally priced against the single agent from Day 1.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- the specialists
# Deterministic stand-ins. Each takes the run state and returns a PARTIAL state -- exactly
# the LangGraph node shape from Module 3 -- and reports what it spent. No model is called,
# so a graph's structure AND its cost can be graded offline and exactly. The "Run it for
# real" cells put the sandbox model behind the same interface.

SANCTIONS_WATCH = {"NORTHWIND"}

COST = {"supervisor": 120, "ledger": 380, "policy": 420, "sanctions": 90, "writer": 610}


def agent_ledger(state: dict) -> dict:
    """Read the payment named in the state."""
    ref = state.get("ref")
    record = LEDGER.get(ref)
    if record is None:
        return {"problems": [f"no payment on file with reference {ref!r}"],
                "tokens": COST["ledger"]}
    return {"facts": {"ref": ref, **record},
            "findings": [{"by": "ledger", "source": "ledger",
                          "claim": f"{ref} is {record['status']} "
                                   f"for {record['amount']:,.2f} {record['ccy']}"}],
            "tokens": COST["ledger"]}


def agent_policy(state: dict) -> dict:
    """Say what the operating policy is for whatever went wrong."""
    code = (state.get("facts") or {}).get("reason_code")
    if code is None:
        return {"problems": ["policy ran before the reason code existed"],
                "tokens": COST["policy"]}
    return {"findings": [{"by": "policy", "source": "policy",
                          "claim": POLICY.get(code, f"no policy on file for {code}")}],
            "needs_human": code in NEEDS_HUMAN,
            "tokens": COST["policy"]}


def agent_sanctions(state: dict) -> dict:
    """A set-membership test. No model needed, and none used -- note the cost column."""
    counterparty = (state.get("facts") or {}).get("counterparty")
    listed = counterparty in SANCTIONS_WATCH
    return {"findings": [{"by": "sanctions", "source": "watchlist",
                          "claim": f"{counterparty} is "
                                   f"{'ON the watchlist' if listed else 'not on the watchlist'}"}],
            "blocked": listed,
            "tokens": COST["sanctions"]}


def agent_writer(state: dict) -> dict:
    """Turn whatever findings arrived into one recommendation."""
    findings = state.get("findings") or []
    if (state.get("facts") or {}).get("status") == "settled":
        action = "no action"                      # nothing to release; it already went
    elif state.get("blocked") or state.get("needs_human"):
        action = "hold for a human"
    else:
        action = "release"
    return {"recommendation": action,
            "rationale": [f["claim"] for f in findings],
            "tokens": COST["writer"]}


AGENTS = {"ledger": agent_ledger, "policy": agent_policy,
          "sanctions": agent_sanctions, "writer": agent_writer}
print("specialists:", ", ".join(AGENTS))

## Concept

Two ideas that get run together and should not be:

- **Decomposition** is a dependency graph. It tells you what *may* run at the same time. Nothing
  about wanting four agents makes four agents able to start.
- **Distribution** is a handoff. It carries exactly what you put in the message &mdash; and by default
  it drops the reasoning, the constraints and the failure history.

The second one produces the bug in Section 4, where both agents behave correctly and the
recommendation is still wrong.

## Section 1 &mdash; What may run at once

Level the dependency graph into waves. Everything in one wave is independent; the number of waves
is the critical path, and no amount of parallelism shortens it.

In [ ]:
TASKS = {
    "read":      {"agent": "ledger",    "needs": []},
    "policy":    {"agent": "policy",    "needs": ["read"]},
    "screen":    {"agent": "sanctions", "needs": ["read"]},
    "recommend": {"agent": "writer",    "needs": ["policy", "screen"]},
}

def waves(tasks: dict) -> list:
    """Group tasks into execution waves. Everything within a wave may run at the same time."""
    done, out, remaining = set(), [], dict(tasks)
    while remaining:
        ready = [name for name, t in remaining.items()
                 if all(need in done for need in t["needs"])]
        if not ready:
            raise ValueError(f"circular dependency among {sorted(remaining)}")
        out.append(sorted(ready))
        done |= set(ready)
        for name in ready:
            remaining.pop(name)
    return out

In [ ]:
# --- Self-check: Section 1
_cycle = {"a": {"agent": "ledger", "needs": ["b"]}, "b": {"agent": "policy", "needs": ["a"]}}

check("four tasks resolve into three waves",
      lambda: len(waves(TASKS)) == 3)
check("nothing can start before the payment is read",
      lambda: waves(TASKS)[0] == ["read"])
check("policy and screening are independent, so they share a wave",
      lambda: waves(TASKS)[1] == ["policy", "screen"])
check("the recommendation waits for both",
      lambda: waves(TASKS)[2] == ["recommend"])
check("every task appears exactly once",
      lambda: sorted(t for w in waves(TASKS) for t in w) == sorted(TASKS))
check("the widest wave is two, so four agents never run four-abreast",
      lambda: max(len(w) for w in waves(TASKS)) == 2,
      "the critical path is three hops whatever you spend on parallelism")
def guard_raises(fn, exc) -> bool:
    """True if fn() raises exc, False if it raises anything else or nothing at all.

    NameError is deliberately re-raised: a helper that swallows it turns an unfilled
    blank into a [FAIL] instead of a [TODO], which is a lie about what went wrong.
    """
    try:
        fn()
    except NameError:
        raise
    except exc:
        return True
    except Exception:
        return False
    return False

check("a circular dependency is refused rather than looping forever",
      lambda: guard_raises(lambda: waves(_cycle), ValueError))

## Section 2 &mdash; Run it

Fold each agent's partial state into the run state, wave by wave. This merge is deliberately
simple: within a wave the agents here write different keys, so nothing collides. Lab 5.3 removes
that comfort.

In [ ]:
def merge(state: dict, partial: dict) -> dict:
    """Fold one agent's partial state into the run state.

    Lists accumulate, counters add, everything else is replaced. That last rule is the one
    that goes wrong once two agents run at the same time -- which is Lab 5.3.
    """
    out = dict(state)
    for key, value in partial.items():
        if key == "tokens":
            out["tokens"] = out.get("tokens", 0) + value
        elif key in ("findings", "problems"):
            out[key] = (out.get(key) or []) + value
        else:
            out[key] = value
    return out


def run_waves(ref: str, tasks: dict = None) -> dict:
    """Run every wave in order, threading the state through."""
    tasks = TASKS if tasks is None else tasks
    state, order = {"ref": ref, "tokens": 0}, []
    for wave in waves(tasks):
        for name in wave:
            state = merge(state, AGENTS[tasks[name]["agent"]](state))
            order.append(name)
    return {"state": state, "order": order, "waves": waves(tasks)}


def _show():
    out = run_waves("PMT-1005")
    print("  waves:", out["waves"])
    print("  spend:", out["state"]["tokens"], "tokens")
    print("  says :", out["state"].get("recommendation"))
    for f in out["state"]["findings"]:
        print(f"    [{f['by']:9} src={f['source']:9}] {f['claim'][:60]}")
guard(_show)

In [ ]:
# --- Self-check: Section 2
def run1005():
    return run_waves("PMT-1005")["state"]

check("all four tasks ran",
      lambda: len(run_waves("PMT-1005")["order"]) == 4)
check("and in dependency order -- read first, recommend last",
      lambda: run_waves("PMT-1005")["order"][0] == "read"
              and run_waves("PMT-1005")["order"][-1] == "recommend")
check("three specialists each contributed a finding",
      lambda: len(run1005()["findings"]) == 3)
check("the watchlisted counterparty was caught",
      lambda: run1005()["blocked"] is True)
check("and the reason code independently demands a human",
      lambda: run1005()["needs_human"] is True)
check("so the recommendation is to hold",
      lambda: run1005()["recommendation"] == "hold for a human")
check("the bill is the sum of what each specialist spent",
      lambda: run1005()["tokens"] == COST["ledger"] + COST["policy"]
                                   + COST["sanctions"] + COST["writer"])
check("a settled payment needs no action",
      lambda: run_waves("PMT-1001")["state"]["recommendation"] == "no action")
check("a payment that does not exist does not crash the run",
      lambda: "problems" in run_waves("PMT-0000")["state"],
      "the whole graph must survive one specialist finding nothing")

## Section 3 &mdash; The handoff payload

A handoff feels like passing a task. What actually crosses is whatever you put in the message,
and nothing else. Build it.

In [ ]:
def handoff(state: dict, to_agent: str, task: str) -> dict:
    """The message one agent sends another. ONLY what is in this dict crosses."""
    message = {"to": to_agent, "task": task, "facts": state.get("facts")}
    message.update({"constraints":   state.get("constraints") or [],
                    "already_tried": state.get("already_tried") or []})
    return message

In [ ]:
# --- Self-check: Section 3
_rich = {"ref": "PMT-1005",
         "facts": {"ref": "PMT-1005", "reason_code": "SANCTIONS_REVIEW"},
         "constraints": ["Do not release PMT-1005 without a human decision"],
         "already_tried": ["auto-retry failed at 09:14"]}

check("the task and the facts cross",
      lambda: handoff(_rich, "policy", "decide")["task"] == "decide"
              and handoff(_rich, "policy", "decide")["facts"]["reason_code"] == "SANCTIONS_REVIEW")
check("the constraints cross",
      lambda: handoff(_rich, "policy", "decide")["constraints"] ==
              ["Do not release PMT-1005 without a human decision"])
check("the failure history crosses, so the next agent does not retry it",
      lambda: handoff(_rich, "policy", "decide")["already_tried"] == ["auto-retry failed at 09:14"])
check("a state with no constraints still hands over the key, empty",
      lambda: handoff({"facts": {}}, "policy", "x")["constraints"] == [],
      "a missing key and an empty list read very differently to the code on the other side")
check("the receiving agent is named",
      lambda: handoff(_rich, "sanctions", "screen")["to"] == "sanctions")

## Section 4 &mdash; Both agents right, answer wrong

Triage establishes that a payment must not be released. It hands off. The policy agent recommends
releasing it. Neither agent malfunctioned.

Run it both ways and watch the constraint decide the outcome.

In [ ]:
def policy_from_handoff(message: dict) -> dict:
    """A policy agent that knows only what the handoff told it -- which is the realistic case."""
    constraints = message.get("constraints") or []
    if any("do not release" in c.lower() for c in constraints):
        return {"recommendation": "hold for a human", "why": "a constraint forbids release"}
    code = (message.get("facts") or {}).get("reason_code")
    return {"recommendation": "release", "why": POLICY.get(code, "no policy on file")}


def investigate(ref: str, carry_constraints: bool = True) -> dict:
    """Triage reads the payment, sets a constraint if one applies, then hands off."""
    state = merge({"ref": ref, "tokens": 0}, agent_ledger({"ref": ref}))
    if (state.get("facts") or {}).get("reason_code") in NEEDS_HUMAN:
        state["constraints"] = [f"Do not release {ref} without a human decision"]
    message = handoff(state, "policy", f"decide whether {ref} can be released")
    if not carry_constraints:
        message.pop("constraints", None)    # the bug, made explicit
    return policy_from_handoff(message)

In [ ]:
# --- Self-check: Section 4
check("carrying the constraint, the sanctions case is held",
      lambda: investigate("PMT-1005")["recommendation"] == "hold for a human")
check("DROPPING IT, the very same case is released",
      lambda: investigate("PMT-1005", carry_constraints=False)["recommendation"] == "release",
      "this is the bug: nothing errored, and both agents did exactly what they were asked")
check("the two runs disagree on the same payment",
      lambda: investigate("PMT-1005")["recommendation"]
              != investigate("PMT-1005", carry_constraints=False)["recommendation"])
check("the limit-breach case is protected the same way",
      lambda: investigate("PMT-1003")["recommendation"] == "hold for a human")
check("a case with no constraint is unaffected either way",
      lambda: investigate("PMT-1002")["recommendation"]
              == investigate("PMT-1002", carry_constraints=False)["recommendation"],
      "the dropped constraint only changes the cases where a constraint existed -- which is why it hides")
check("the held recommendation says which constraint stopped it",
      lambda: "constraint" in investigate("PMT-1005")["why"])

def _both_ways():
    for ref in ("PMT-1005", "PMT-1003", "PMT-1002"):
        with_c = investigate(ref)["recommendation"]
        without = investigate(ref, carry_constraints=False)["recommendation"]
        flag = "  <-- DIFFERENT" if with_c != without else ""
        print(f"  {ref}   carried: {with_c:18} dropped: {without:18}{flag}")
guard(_both_ways)

## Run it for real

Give the model the two handoff messages &mdash; one with the constraint, one without &mdash; and ask it
for a recommendation. It is not being tested. Your message is.

In [ ]:
if llm_ready():
    def _ask_both():
        state = merge({"ref": "PMT-1005", "tokens": 0}, agent_ledger({"ref": "PMT-1005"}))
        state["constraints"] = ["Do not release PMT-1005 without a human decision"]
        full = handoff(state, "policy", "decide whether PMT-1005 can be released")
        thin = {k: v for k, v in full.items() if k != "constraints"}
        for label, message in (("with constraint", full), ("without      ", thin)):
            reply = ask("You are the policy agent. Given this handoff, reply in one sentence with "
                        "your recommendation.\n\n" + json.dumps(message, default=str))
            print(f"  [{label}] {reply.strip()[:180]}")
            print()
    guard(_ask_both)

### Read it

If the two replies differ, you have watched a correct agent reach a wrong conclusion because of
what it was not told. No prompt engineering fixes that, and no stronger model does either &mdash; the
information was not in the room.

**The rule:** a handoff carries the task, the findings, the constraints and what has already been
tried. Three of those four are the ones people forget, and each has its own signature bug &mdash;
paying twice, breaking a rule it never saw, and retrying what already failed.

In [ ]:
score()

## Your turn

1. Add the third dropped thing &mdash; the reasoning &mdash; and measure what carrying it costs in
   tokens against what re-deriving it costs. One of those is a bill and one is a risk.
2. `waves` assumes every task in a wave succeeds. Make `screen` fail and decide what wave 3 should
   receive: a partial state, or nothing at all.
3. Turn `TASKS` into two graphs &mdash; one for high-value payments and one for low &mdash; and let
   Lab 5.1's supervisor choose between them. That is routing by value, and Lab 5.5 prices it.